In [1]:
import json
from pathlib import Path
from collections import Counter

INPUT_PATH = Path("../../data/processed/eval/eval_dataset.json")
OUTPUT_PATH = Path("../../data/processed/eval/eval_dataset_cleaned.json")
REPORT_PATH = Path("../../data/processed/eval/eval_dataset_cleaning_report.json")


def is_nan_like(value) -> bool:
    """
    문자열/값이 nan, None, 빈 문자열 계열인지 확인합니다.
    """
    if value is None:
        return True

    value_str = str(value).strip().lower()

    return value_str in {
        "",
        "nan",
        "none",
        "null",
        "na",
        "n/a"
    }


def has_nan_gold_id(gold_ids) -> bool:
    """
    gold_ids 안에 nan 계열 값이 있는지 확인합니다.
    """
    if not isinstance(gold_ids, list) or len(gold_ids) == 0:
        return True

    return any(is_nan_like(gold_id) for gold_id in gold_ids)


def is_valid_keyword_groups(keyword_groups) -> bool:
    """
    required_keyword_groups가 평가 가능한 형태인지 확인합니다.
    기대 형태:
    [
        ["키워드1", "키워드2"],
        ["키워드3"]
    ]
    """
    if not isinstance(keyword_groups, list) or len(keyword_groups) == 0:
        return False

    for group in keyword_groups:
        if not isinstance(group, list) or len(group) == 0:
            return False

        valid_keywords = [
            keyword for keyword in group
            if not is_nan_like(keyword)
        ]

        if len(valid_keywords) == 0:
            return False

    return True


def clean_eval_dataset(input_path: Path, output_path: Path, report_path: Path):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    original_count = len(data)

    qid_counter = Counter()
    cleaned = []
    removed = []

    for idx, item in enumerate(data):
        qid = item.get("qid")
        question = item.get("question")
        reference = item.get("reference")
        gold_ids = item.get("gold_ids")
        required_keyword_groups = item.get("required_keyword_groups")

        remove_reasons = []

        # 1. qid 검사
        if is_nan_like(qid):
            remove_reasons.append("invalid_qid")

        elif "nan" in str(qid).lower():
            remove_reasons.append("qid_contains_nan")

        # 2. question 검사
        if is_nan_like(question):
            remove_reasons.append("empty_question")

        # 3. reference 검사
        if is_nan_like(reference):
            remove_reasons.append("empty_reference")

        # 4. gold_ids 검사
        if has_nan_gold_id(gold_ids):
            remove_reasons.append("invalid_or_nan_gold_ids")

        # 5. required_keyword_groups 검사
        if not is_valid_keyword_groups(required_keyword_groups):
            remove_reasons.append("invalid_required_keyword_groups")

        # 6. qid 중복 검사
        if not is_nan_like(qid):
            qid_counter[str(qid)] += 1

            if qid_counter[str(qid)] > 1:
                remove_reasons.append("duplicated_qid")

        if remove_reasons:
            removed.append({
                "index": idx,
                "qid": qid,
                "question": question,
                "gold_ids": gold_ids,
                "remove_reasons": remove_reasons
            })
            continue

        cleaned.append(item)

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)

    reason_counter = Counter()
    for row in removed:
        for reason in row["remove_reasons"]:
            reason_counter[reason] += 1

    report = {
        "input_path": str(input_path),
        "output_path": str(output_path),
        "original_count": original_count,
        "cleaned_count": len(cleaned),
        "removed_count": len(removed),
        "remove_reason_counts": dict(reason_counter),
        "removed_samples": removed[:30]
    }

    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print("평가 데이터셋 정리 완료")
    print(f"- 원본 문항 수: {original_count}")
    print(f"- 정리 후 문항 수: {len(cleaned)}")
    print(f"- 제거 문항 수: {len(removed)}")
    print(f"- 저장 파일: {output_path}")
    print(f"- 리포트 파일: {report_path}")

    print("\n제거 사유별 개수")
    for reason, count in reason_counter.items():
        print(f"- {reason}: {count}")

    return cleaned, report


cleaned_data, report = clean_eval_dataset(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    report_path=REPORT_PATH
)

평가 데이터셋 정리 완료
- 원본 문항 수: 391
- 정리 후 문항 수: 327
- 제거 문항 수: 64
- 저장 파일: ../../data/processed/eval/eval_dataset_cleaned.json
- 리포트 파일: ../../data/processed/eval/eval_dataset_cleaning_report.json

제거 사유별 개수
- qid_contains_nan: 64
- invalid_or_nan_gold_ids: 64
- duplicated_qid: 60


In [3]:
import json
import pandas as pd
from pathlib import Path


EVAL_INPUT_PATH = Path("../../data/processed/eval/eval_dataset_cleaned.json")
DATA_LIST_PATH = Path("../../data/raw/data_list.csv")

EVAL_OUTPUT_PATH = Path("../../data/processed/eval/eval_dataset_v2_no_deadline.json")
REPORT_OUTPUT_PATH = Path("../../data/processed/eval/eval_dataset_v2_report.json")


def infer_question_type(qid: str) -> str:
    qid = str(qid)

    if "fact_budget" in qid:
        return "fact_budget"
    if "fact_deadline" in qid:
        return "fact_deadline"
    if "llm_1" in qid:
        return "llm_1"
    if "llm_2" in qid:
        return "llm_2"

    return "unknown"


def infer_source_type(question_type: str) -> str:
    if question_type == "fact_budget":
        return "metadata"

    if question_type in {"llm_1", "llm_2"}:
        return "document"

    return "unknown"


def infer_answer_format(question_type: str, question: str) -> str:
    question = str(question)

    if question_type == "fact_budget":
        return "money"

    if "연락처" in question or "전화번호" in question or "담당자" in question:
        return "contact"

    if "기간" in question or "사업기간" in question or "과업기간" in question:
        return "period"

    if "범위" in question or "항목" in question or "무엇" in question:
        return "list"

    return "summary"


def normalize_id(value) -> str:
    return str(value).strip()


def build_metadata_map(data_list_path: Path) -> dict:
    """
    data_list.csv에서 공고 번호 기준 메타데이터 매핑을 만듭니다.
    공고 번호가 없는 행은 이미 제거했다고 가정합니다.
    """
    df = pd.read_csv(data_list_path)

    metadata_map = {}

    for idx, row in df.iterrows():
        notice_no = row.get("공고 번호")

        if pd.isna(notice_no):
            continue

        doc_id = normalize_id(notice_no)

        metadata_map[doc_id] = {
            "source_row_id": int(idx),
            "project_name": row.get("사업명", ""),
            "organization": row.get("발주 기관", ""),
            "file_name": row.get("파일명", ""),
            "file_type": row.get("파일형식", "")
        }

    return metadata_map


def enrich_eval_dataset(
    eval_input_path: Path,
    data_list_path: Path,
    eval_output_path: Path,
    report_output_path: Path
):
    with open(eval_input_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)

    metadata_map = build_metadata_map(data_list_path)

    enriched = []
    removed = []

    for item in eval_data:
        qid = item.get("qid", "")
        question = item.get("question", "")
        gold_ids = item.get("gold_ids", [])

        question_type = item.get("question_type") or infer_question_type(qid)

        # 1. fact_deadline 제거
        if question_type == "fact_deadline" or "fact_deadline" in str(qid):
            removed.append({
                "qid": qid,
                "reason": "remove_fact_deadline"
            })
            continue

        # 2. doc_id 추가
        doc_id = item.get("doc_id")

        if not doc_id:
            if isinstance(gold_ids, list) and len(gold_ids) > 0:
                doc_id = normalize_id(gold_ids[0])
            else:
                removed.append({
                    "qid": qid,
                    "reason": "missing_doc_id_and_gold_ids"
                })
                continue

        # 3. 메타데이터 보강
        meta = metadata_map.get(doc_id, {})

        item["doc_id"] = doc_id
        item["question_type"] = question_type
        item["source_type"] = item.get("source_type") or infer_source_type(question_type)
        item["answer_format"] = item.get("answer_format") or infer_answer_format(question_type, question)

        item["source_row_id"] = item.get("source_row_id", meta.get("source_row_id"))
        item["project_name"] = item.get("project_name") or meta.get("project_name", "")
        item["organization"] = item.get("organization") or meta.get("organization", "")
        item["file_name"] = item.get("file_name") or meta.get("file_name", "")
        item["file_type"] = item.get("file_type") or meta.get("file_type", "")

        enriched.append(item)

    eval_output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(eval_output_path, "w", encoding="utf-8") as f:
        json.dump(enriched, f, ensure_ascii=False, indent=2)

    report = {
        "input_path": str(eval_input_path),
        "output_path": str(eval_output_path),
        "original_count": len(eval_data),
        "final_count": len(enriched),
        "removed_count": len(removed),
        "removed_items": removed[:50],
        "metadata_matched_count": sum(1 for item in enriched if item.get("project_name")),
        "metadata_unmatched_count": sum(1 for item in enriched if not item.get("project_name"))
    }

    with open(report_output_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print("평가 데이터셋 v2 생성 완료")
    print(f"- 원본 문항 수: {len(eval_data)}")
    print(f"- 최종 문항 수: {len(enriched)}")
    print(f"- 제거 문항 수: {len(removed)}")
    print(f"- 메타데이터 매칭 성공: {report['metadata_matched_count']}")
    print(f"- 메타데이터 매칭 실패: {report['metadata_unmatched_count']}")
    print(f"- 저장 파일: {eval_output_path}")
    print(f"- 리포트 파일: {report_output_path}")

    return enriched, report


enriched_data, report = enrich_eval_dataset(
    eval_input_path=EVAL_INPUT_PATH,
    data_list_path=DATA_LIST_PATH,
    eval_output_path=EVAL_OUTPUT_PATH,
    report_output_path=REPORT_OUTPUT_PATH
)

평가 데이터셋 v2 생성 완료
- 원본 문항 수: 327
- 최종 문항 수: 245
- 제거 문항 수: 82
- 메타데이터 매칭 성공: 245
- 메타데이터 매칭 실패: 0
- 저장 파일: ../../data/processed/eval/eval_dataset_v2_no_deadline.json
- 리포트 파일: ../../data/processed/eval/eval_dataset_v2_report.json


In [5]:
import json
from pathlib import Path

INPUT_PATH = Path("../../data/processed/eval/eval_dataset_v2_no_deadline.json")
OUTPUT_PATH = Path("../../data/processed/eval/eval_dataset_v3.json")


def remove_gold_keyword_groups(input_path: Path, output_path: Path):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    removed_count = 0

    for item in data:
        if "gold_keyword_groups" in item:
            del item["gold_keyword_groups"]
            removed_count += 1

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print("gold_keyword_groups 제거 완료")
    print(f"- 전체 문항 수: {len(data)}")
    print(f"- 제거된 필드 수: {removed_count}")
    print(f"- 저장 파일: {output_path}")

    return data


data = remove_gold_keyword_groups(INPUT_PATH, OUTPUT_PATH)

gold_keyword_groups 제거 완료
- 전체 문항 수: 245
- 제거된 필드 수: 245
- 저장 파일: ../../data/processed/eval/eval_dataset_v3.json


In [6]:
import json
from pathlib import Path

INPUT_PATH = Path("../../data/processed/eval/eval_dataset_v3.json")
OUTPUT_PATH = Path("../../data/processed/eval/eval_dataset_v4_final.json")


def remove_gold_ids(input_path: Path, output_path: Path):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    removed_count = 0
    missing_doc_id_count = 0

    for item in data:
        # doc_id가 없으면 기존 gold_ids[0]로 보완
        if not item.get("doc_id"):
            gold_ids = item.get("gold_ids", [])
            if isinstance(gold_ids, list) and len(gold_ids) > 0:
                item["doc_id"] = str(gold_ids[0])
            else:
                missing_doc_id_count += 1

        # gold_ids 제거
        if "gold_ids" in item:
            del item["gold_ids"]
            removed_count += 1

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print("gold_ids 제거 완료")
    print(f"- 전체 문항 수: {len(data)}")
    print(f"- 제거된 gold_ids 필드 수: {removed_count}")
    print(f"- doc_id 누락 문항 수: {missing_doc_id_count}")
    print(f"- 저장 파일: {output_path}")

    return data


data = remove_gold_ids(INPUT_PATH, OUTPUT_PATH)

gold_ids 제거 완료
- 전체 문항 수: 245
- 제거된 gold_ids 필드 수: 245
- doc_id 누락 문항 수: 0
- 저장 파일: ../../data/processed/eval/eval_dataset_v4_final.json


In [7]:
import json
import re
from pathlib import Path

INPUT_PATH = Path("../../data/processed/eval/eval_dataset_v4_final.json")
OUTPUT_PATH = Path("../../data/processed/eval/eval_dataset_v5_budget_keywords.json")


def format_won_with_commas(amount: int) -> str:
    return f"{amount:,}원"


def format_won_no_commas(amount: int) -> str:
    return f"{amount}원"


def format_korean_money(amount: int) -> str:
    """
    130000000 -> 1억3천만원
    150000000 -> 1억5천만원
    11270000000 -> 112억7천만원
    0 -> 0원

    평가용 키워드 생성 목적이므로 자연스러운 축약 표현 위주로 생성합니다.
    """
    if amount == 0:
        return "0원"

    eok = amount // 100_000_000
    remainder = amount % 100_000_000

    man = remainder // 10_000
    cheon_man = man // 1000
    baek_man = (man % 1000) // 100
    ship_man = (man % 100) // 10
    il_man = man % 10

    parts = []

    if eok > 0:
        parts.append(f"{eok}억")

    # 1천만원 단위가 있으면 "3천만원"
    if cheon_man > 0:
        parts.append(f"{cheon_man}천")

    # 1백만원 단위가 있으면 "2백만원"
    if baek_man > 0:
        parts.append(f"{baek_man}백")

    # 10만원 단위 이하까지는 너무 세밀해서 평가 키워드로는 원 숫자 표기를 우선 사용
    if ship_man > 0:
        parts.append(f"{ship_man}십")

    if il_man > 0:
        parts.append(f"{il_man}")

    # 만원 단위 표현이 있으면 마지막에 "만원" 부착
    if man > 0:
        if parts and not parts[-1].endswith("억"):
            parts[-1] = parts[-1] + "만원"
        elif eok == 0:
            parts.append("만원")

    return "".join(parts)


def extract_amount_from_keyword_groups(required_keyword_groups):
    """
    required_keyword_groups에서 숫자만 있는 값을 찾아 금액 int로 변환합니다.
    예: [["130000000"]] -> 130000000
    """
    if not isinstance(required_keyword_groups, list):
        return None

    for group in required_keyword_groups:
        if not isinstance(group, list):
            continue

        for keyword in group:
            keyword_str = str(keyword)
            digits = re.sub(r"[^0-9]", "", keyword_str)

            if digits:
                return int(digits)

    return None


def expand_budget_keywords(input_path: Path, output_path: Path):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    changed_count = 0
    skipped_count = 0

    for item in data:
        question_type = item.get("question_type", "")
        answer_format = item.get("answer_format", "")

        is_budget_item = (
            question_type == "fact_budget"
            or answer_format == "money"
        )

        if not is_budget_item:
            continue

        amount = extract_amount_from_keyword_groups(
            item.get("required_keyword_groups", [])
        )

        if amount is None:
            skipped_count += 1
            continue

        expanded_keywords = [
            format_won_with_commas(amount),
            format_won_no_commas(amount),
            format_korean_money(amount)
        ]

        # 중복 제거하되 순서 유지
        expanded_keywords = list(dict.fromkeys(expanded_keywords))

        item["required_keyword_groups"] = [expanded_keywords]
        changed_count += 1

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print("fact_budget required_keyword_groups 확장 완료")
    print(f"- 전체 문항 수: {len(data)}")
    print(f"- 수정된 fact_budget/money 문항 수: {changed_count}")
    print(f"- 금액 추출 실패 문항 수: {skipped_count}")
    print(f"- 저장 파일: {output_path}")

    return data


data = expand_budget_keywords(INPUT_PATH, OUTPUT_PATH)

fact_budget required_keyword_groups 확장 완료
- 전체 문항 수: 245
- 수정된 fact_budget/money 문항 수: 81
- 금액 추출 실패 문항 수: 0
- 저장 파일: ../../data/processed/eval/eval_dataset_v5_budget_keywords.json


In [8]:
import json
import re
from pathlib import Path

INPUT_PATH = Path("../../data/processed/eval/eval_dataset_v5.json")
OUTPUT_PATH = Path("../../data/processed/eval/eval_dataset_v6.json")
REPORT_PATH = Path("../../data/processed/eval/eval_dataset_v6_filter_report.json")


def extract_amount_from_reference(reference: str) -> int | None:
    """
    reference에서 금액 숫자만 추출합니다.
    예:
    - "130,000,000원 입니다." -> 130000000
    - "0원 입니다." -> 0
    - "1원 입니다." -> 1
    """
    if reference is None:
        return None

    digits = re.sub(r"[^0-9]", "", str(reference))

    if not digits:
        return None

    return int(digits)


def remove_abnormal_budget_items(
    input_path: Path,
    output_path: Path,
    report_path: Path,
    min_valid_budget: int = 10_000_000
):
    """
    fact_budget 문항 중 사업 배정 예산이 min_valid_budget 미만인 항목을 제거합니다.

    기본값:
    - 10,000,000원 미만 제거

    이유:
    - RFP/공공·기업 용역 사업 예산이 0원, 1원 등으로 들어간 경우는
      실제 예산이라기보다 메타데이터 결측/오류일 가능성이 높음.
    """
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    kept = []
    removed = []

    for item in data:
        question_type = item.get("question_type")
        reference = item.get("reference", "")

        if question_type == "fact_budget":
            amount = extract_amount_from_reference(reference)

            if amount is None:
                removed.append({
                    "qid": item.get("qid"),
                    "doc_id": item.get("doc_id"),
                    "project_name": item.get("project_name"),
                    "reference": reference,
                    "reason": "budget_amount_not_found"
                })
                continue

            if amount < min_valid_budget:
                removed.append({
                    "qid": item.get("qid"),
                    "doc_id": item.get("doc_id"),
                    "project_name": item.get("project_name"),
                    "reference": reference,
                    "amount": amount,
                    "reason": f"budget_less_than_{min_valid_budget}"
                })
                continue

        kept.append(item)

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(kept, f, ensure_ascii=False, indent=2)

    report = {
        "input_path": str(input_path),
        "output_path": str(output_path),
        "original_count": len(data),
        "final_count": len(kept),
        "removed_count": len(removed),
        "min_valid_budget": min_valid_budget,
        "removed_items": removed
    }

    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print("비정상 예산 문항 제거 완료")
    print(f"- 원본 문항 수: {len(data)}")
    print(f"- 최종 문항 수: {len(kept)}")
    print(f"- 제거 문항 수: {len(removed)}")
    print(f"- 최소 유효 예산 기준: {min_valid_budget:,}원")
    print(f"- 저장 파일: {output_path}")
    print(f"- 리포트 파일: {report_path}")

    print("\n제거된 문항")
    for row in removed:
        print(f"- {row['qid']} | {row.get('reference')} | {row.get('project_name')}")

    return kept, report


filtered_data, report = remove_abnormal_budget_items(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    report_path=REPORT_PATH,
    min_valid_budget=10_000_000
)

비정상 예산 문항 제거 완료
- 원본 문항 수: 245
- 최종 문항 수: 243
- 제거 문항 수: 2
- 최소 유효 예산 기준: 10,000,000원
- 저장 파일: ../../data/processed/eval/eval_dataset_v6.json
- 리포트 파일: ../../data/processed/eval/eval_dataset_v6_filter_report.json

제거된 문항
- 20241138864_fact_budget | 0원 입니다. | 을지대학교 비교과시스템 개발
- 20240311752_fact_budget | 1원 입니다. | 실손보험 청구 전산화 시스템 구축 사업
